# Day11：Human in the Loop

本 Notebook 不调用 LLM，演示只读提案、LangGraph 中断、批准应用、拒绝和源文件冲突。生产实现位于 `memory/approval.py`、`tools/approval_tool.py`、`workflow/human_approval.py` 与 `workflow/runtime.py`。

In [ ]:
import os
import sqlite3
import tempfile
from typing import Any, Dict, TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

from memory.approval import ApprovalStore
from memory.patch_history import PatchHistory
from tools.approval_tool import ApprovalTool
from tools.change_proposal_tool import ChangeProposalTool
from tools.diff_tool import DiffTool
from tools.file_manager import FileManager
from workflow.human_approval import ChangeProposalNode, HumanApprovalNode

temporary_directory = tempfile.TemporaryDirectory()
generated_root = os.path.join(temporary_directory.name, 'generated')
os.makedirs(generated_root)
file_manager = FileManager()
diff_tool = DiffTool(file_manager, generated_root)
approval_store = ApprovalStore(os.path.join(temporary_directory.name, 'approvals.json'))
proposal_tool = ChangeProposalTool(file_manager, generated_root, diff_tool)
approval_tool = ApprovalTool(
    approval_store,
    diff_tool,
    PatchHistory(os.path.join(temporary_directory.name, 'patches.json'), diff_tool),
)
print(generated_root)

In [ ]:
proposal = proposal_tool.propose(
    [{'file': 'Preview.cs', 'content': 'class Preview {}\n'}],
    source='coder',
)
assert proposal['patches']
assert not os.path.exists(os.path.join(generated_root, 'Preview.cs'))
print(proposal['patches'][0]['diff'])

In [ ]:
class ApprovalState(TypedDict, total=False):
    proposal_source: str
    proposed_changes: list
    approval_request: Dict[str, Any]
    approval_result: Dict[str, Any]
    approval_history: list
    approval_status: str
    code: list
    current_agent: str

proposal_node = ChangeProposalNode(proposal_tool, approval_store)
approval_node = HumanApprovalNode(approval_tool)
builder = StateGraph(ApprovalState)
builder.add_node('proposal', proposal_node.run)
builder.add_node('approval', approval_node.run)
builder.add_edge(START, 'proposal')
builder.add_edge('proposal', 'approval')
builder.add_edge('approval', END)
connection = sqlite3.connect(
    os.path.join(temporary_directory.name, 'workflow.sqlite'),
    check_same_thread=False,
)
graph = builder.compile(checkpointer=SqliteSaver(connection))

In [ ]:
approve_config = {'configurable': {'thread_id': 'approve-demo'}}
interrupted = graph.invoke(
    {
        'proposal_source': 'coder',
        'proposed_changes': [{'file': 'Approved.cs', 'content': 'class Approved {}\n'}],
        'approval_history': [],
        'code': [{'file': 'Approved.cs', 'content': 'class Approved {}\n'}],
    },
    config=approve_config,
)
request = interrupted['__interrupt__'][0].value
assert request['status'] == 'pending'
approved = graph.invoke(
    Command(resume={'bundle_id': request['bundle_id'], 'action': 'approve', 'mode': 'batch'}),
    config=approve_config,
)
assert approved['approval_status'] == 'approved'
assert os.path.isfile(os.path.join(generated_root, 'Approved.cs'))
print('approved:', request['bundle_id'])

In [ ]:
reject_config = {'configurable': {'thread_id': 'reject-demo'}}
interrupted = graph.invoke(
    {
        'proposal_source': 'repair',
        'proposed_changes': [{'file': 'Rejected.cs', 'content': 'class Rejected {}\n'}],
        'approval_history': [],
    },
    config=reject_config,
)
request = interrupted['__interrupt__'][0].value
rejected = graph.invoke(
    Command(resume={'bundle_id': request['bundle_id'], 'action': 'reject', 'mode': 'batch'}),
    config=reject_config,
)
assert rejected['approval_status'] == 'rejected'
assert not os.path.exists(os.path.join(generated_root, 'Rejected.cs'))
print('rejected:', request['bundle_id'])

In [ ]:
conflict_path = os.path.join(generated_root, 'Conflict.cs')
file_manager.write_file(conflict_path, 'before\n')
conflict_config = {'configurable': {'thread_id': 'conflict-demo'}}
interrupted = graph.invoke(
    {
        'proposal_source': 'repair',
        'proposed_changes': [{'file': 'Conflict.cs', 'content': 'approved version\n'}],
        'approval_history': [],
    },
    config=conflict_config,
)
request = interrupted['__interrupt__'][0].value
file_manager.write_file(conflict_path, 'external edit\n')
conflicted = graph.invoke(
    Command(resume={'bundle_id': request['bundle_id'], 'action': 'approve', 'mode': 'batch'}),
    config=conflict_config,
)
assert conflicted['approval_status'] == 'conflicted'
assert file_manager.read_file(conflict_path) == 'external edit\n'
print('conflicted without overwrite:', request['bundle_id'])

In [ ]:
connection.close()
temporary_directory.cleanup()
print('Day11 no-LLM acceptance passed')